In [1]:
#@title 0) Fiksēt bibliotēku versijas un pārbaudīt vidi

!pip install -q \
    stanza==1.12.0 \
    scikit-learn==1.6.1 \
    pandas==2.2.2 \
    numpy==2.0.2 \
    matplotlib==3.10.0 \
    seaborn==0.13.2 \
    openpyxl==3.1.5

import sys
import platform
import importlib.metadata as md

EXPECTED_VERSIONS = {
    "stanza": "1.12.0",
    "scikit-learn": "1.6.1",
    "pandas": "2.2.2",
    "numpy": "2.0.2",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "openpyxl": "3.1.5",
}

print("Python:", sys.version)
print("Platform:", platform.platform())
print()

for package, expected in EXPECTED_VERSIONS.items():
    actual = md.version(package)
    status = "OK" if actual == expected else "ATŠĶIRAS"
    print(f"{package}: {actual} | gaidīts: {expected} | {status}")

for package, expected in EXPECTED_VERSIONS.items():
    actual = md.version(package)
    if actual != expected:
        raise RuntimeError(
            f"Versija nesakrīt: {package}. "
            f"Gaidīts {expected}, bet instalēts {actual}. "
            f"Pārstartē runtime un palaid šo bloku vēlreiz."
        )

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35

stanza: 1.12.0 | gaidīts: 1.12.0 | OK
scikit-learn: 1.6.1 | gaidīts: 1.6.1 | OK
pandas: 2.2.2 | gaidīts: 2.2.2 | OK
numpy: 2.0.2 | gaidīts: 2.0.2 | OK
matplotlib: 3.10.0 | gaidīts: 3.10.0 | OK
seaborn: 0.13.2 | gaidīts: 0.13.2 | OK
openpyxl: 3.1.5 | gaidīts: 3.1.5 | OK


In [2]:
#@title Reproducēt 3. tabulu: korpusa apjoms pa gadiem TF–IDF/NMF modelēšanai

# ============================================================
# 1. Importi
# ============================================================

import os
import re
import zipfile
import pandas as pd
import stanza

try:
    from google.colab import files
    COLAB = True
except Exception:
    COLAB = False

if not COLAB:
    raise RuntimeError("Šis kods paredzēts Google Colab videi.")

# ============================================================
# 2. Ievades faila atrašana
#
# ZIP failu augšupielādē Colab kreisajā Files panelī:
# /content/input/LCLC_1934-1940_only_unique_text_leaflets.zip
# vai vienkārši /content/
# ============================================================

INPUT_DIRS = ["/content/input", "/content"]

ZIP_CANDIDATES = [
    "LCLC_1934-1940_only_unique_text_leaflets.zip",
    "LCLC_1934_1940_only_unique_text_leaflets.zip"
]

ZIP_PATH = None

for d in INPUT_DIRS:
    for z in ZIP_CANDIDATES:
        p = os.path.join(d, z)
        if os.path.exists(p):
            ZIP_PATH = p
            break
    if ZIP_PATH:
        break

# Ja precīzs nosaukums nav atrasts, paņem pirmo .zip failu /content/input vai /content
if ZIP_PATH is None:
    for d in INPUT_DIRS:
        if os.path.exists(d):
            zip_files = [
                os.path.join(d, f)
                for f in os.listdir(d)
                if f.lower().endswith(".zip")
            ]
            if zip_files:
                ZIP_PATH = sorted(zip_files)[0]
                break

if ZIP_PATH is None:
    raise FileNotFoundError(
        "Nav atrasts ZIP fails. Augšupielādē .zip failu Colab Files panelī "
        "mapē /content/input vai /content."
    )

print("Izmantotais ZIP fails:", ZIP_PATH)

# ============================================================
# 3. Konfigurācija
# ============================================================

TARGET_YEARS = list(range(1934, 1940))  # 1934–1939

LATVIAN_STOP_LEMMAS = set(
    "un ir būt tikt kļūt par lai kas bet arī kā tas šis pie pret vai kad tad tik "
    "ar no uz mēs jūs viņš viņa kurš kāds viss jo gan vēl jau pat kurp ne nav "
    "ļoti tikai nekā taču man tevi sev savs".split()
)

META_STOP = {
    "id", "file_name", "title", "author", "date", "source", "text",
    "print_run", "typography_name", "production_method"
}

STOPWORDS = LATVIAN_STOP_LEMMAS | META_STOP

ALLOWED_POS = {"NOUN", "VERB", "ADJ", "PROPN"}

# LCLC vārdu skaitīšanas princips no iepriekšējā skripta
WORD_RE = re.compile(r"\b[\w'’\-]+\b", flags=re.UNICODE)

# ============================================================
# 4. Stanza inicializācija
# ============================================================

print("\nLejupielādē un inicializē Stanza latviešu valodas modeli...")

stanza.download("lv", processors="tokenize,pos,lemma")

nlp = stanza.Pipeline(
    "lv",
    processors="tokenize,pos,lemma",
    use_gpu=True,
    logging_level="WARN"
)

# ============================================================
# 5. Palīgfunkcijas: metadati, text: bloks, gads
# ============================================================

def decode_bytes(raw_bytes: bytes) -> str:
    return raw_bytes.decode("utf-8", errors="replace")


def get_metadata_field(raw_text: str, field_name: str) -> str:
    """
    Nolasa vienas rindas metadatu lauku:
    id:
    file_name:
    date:
    """
    pattern = rf"(?im)^\s*{re.escape(field_name)}\s*:\s*(.+?)\s*$"
    m = re.search(pattern, raw_text)
    return m.group(1).strip() if m else ""


def extract_text_body_for_word_count(raw_bytes: bytes) -> str:
    """
    LCLC vārdu/rakstzīmju skaitīšanai:
    paņem tikai tekstu pēc pirmā 'text:' marķiera.
    Ja 'text:' nav atrasts, izmanto visu faila saturu.
    """
    s = raw_bytes.decode("utf-8", errors="replace")

    m = re.search(r"(?im)^\s*text:\s*\n", s)
    if m:
        return s[m.end():].strip()

    m = re.search(r"(?im)\btext:\s*(.*)", s, flags=re.DOTALL)
    if m:
        return m.group(1).strip()

    return s.strip()


def extract_text_block_for_lemmatization(full_text: str) -> str:
    """
    TF–IDF/NMF priekšapstrādei:
    tā pati loģika, kas izmantota NMF notebook funkcijā extract_text_block().
    """
    parts = re.split(r"(?im)^\s*text\s*:\s*", full_text, maxsplit=1)
    return parts[1].strip() if len(parts) == 2 else full_text.strip()


def extract_id(full_text: str, fallback_filename: str = ""):
    m = re.search(r"(?im)^\s*id\s*:\s*(\d+)", full_text)
    if m:
        return int(m.group(1))

    m2 = re.search(r"^(\d+)", os.path.basename(fallback_filename))
    if m2:
        return int(m2.group(1))

    return None


def extract_year(full_text: str, fallback_filename: str = ""):
    """
    Vispirms meklē gadu date: laukā, pēc tam file_name: laukā,
    pēc tam faktiskajā faila nosaukumā.
    """
    for regex in [
        r"(?im)^\s*date\s*:\s*(.+)$",
        r"(?im)^\s*file_name\s*:\s*(.+)$"
    ]:
        m = re.search(regex, full_text)
        if m:
            y = re.search(r"(19\d{2}|20\d{2})", m.group(1).strip())
            if y:
                return int(y.group(1))

    y3 = re.search(r"(19\d{2}|20\d{2})", fallback_filename)
    if y3:
        return int(y3.group(1))

    return None

# ============================================================
# 6. LCLC vārdu un rakstzīmju skaitīšana
# ============================================================

def count_lclc_word_metrics(text_body: str) -> dict:
    """
    Skaita vārdu un rakstzīmju apjomu pēc LCLC skripta principa.
    Skaitīts tiek tikai pamata teksts pēc text: marķiera.
    """
    return {
        "chars_with_spaces": len(text_body),
        "chars_without_spaces": len(re.sub(r"\s+", "", text_body)),
        "words": len(WORD_RE.findall(text_body)),
        "lines": len(text_body.splitlines()) if text_body else 0
    }

# ============================================================
# 7. TF–IDF/NMF priekšapstrādes lematizācija
# ============================================================

def normalize_lemma(lemma: str) -> str:
    """
    Tā pati manuālās normalizācijas loģika, kas izmantota TF–IDF/NMF notebook.
    """
    lemma = lemma.lower().strip()

    corrections = {
        "cmāte": "māte",
        "madīt": "madāma",
    }

    if lemma in corrections:
        return corrections[lemma]

    prefixes = {
        "ulman": "ulmanis",
        "hitler": "hitlers",
        "latvij": "latvija",
        "sociāldemokr": "sociāldemokrāts",
        "fašist": "fašists",
        "komunist": "komunists",
        "strādniek": "strādnieks",
        "bezdarbniek": "bezdarbnieks",
        "kapitālist": "kapitālists",
        "buržuāz": "buržuāzija",
        "politieslodz": "politieslodzītais",
        "proletariāt": "proletariāts",
    }

    for pref, norm in prefixes.items():
        if lemma.startswith(pref):
            return norm

    return lemma


def lemmatize_text_like_nmf(full_text: str) -> str:
    """
    Atkārto TF–IDF/NMF eksperimenta priekšapstrādi un atgriež clean_text.
    Rezultātā iegūtās vienības pēc tam tiek saskaitītas ar .split().
    """
    text = extract_text_block_for_lemmatization(full_text)

    replacements = {
        r"s\.\-d\.?": " sociāldemokrāti ",
        r"1\.\s*maijs": " pirmais_maijs ",
        r"7\.\s*novembris": " septītais_novembris ",
        r"padomju latvija": " padomju_latvija ",
        r"politiskā pārvalde": " politiskā_pārvalde ",
    }

    for pat, repl in replacements.items():
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)

    # Tāds pats drošības ierobežojums kā NMF notebook
    if len(text) > 100000:
        text = text[:100000]

    # Defisētu vārdu sadalīšana divās vienībās
    text = re.sub(
        r"([A-Za-zĀČĒĢĪĶĻŅŠŪŽāčēģīķļņšūž]+)[-–—]([A-Za-zĀČĒĢĪĶĻŅŠŪŽāčēģīķļņšūž]+)",
        r"\1 \2",
        text
    )

    doc = nlp(text)

    lemmas = []

    for sentence in doc.sentences:
        for word in sentence.words:
            lemma = normalize_lemma(word.lemma or "")

            if len(lemma) <= 2:
                continue

            if lemma in STOPWORDS:
                continue

            if word.upos not in ALLOWED_POS:
                continue

            if not re.fullmatch(r"[a-zāčēģīķļņšūž_]+", lemma):
                continue

            lemmas.append(lemma)

    return " ".join(lemmas)

# ============================================================
# 8. ZIP apstrāde un per-file datu izveide
# ============================================================

rows = []
failed_rows = []

with zipfile.ZipFile(ZIP_PATH, "r") as zip_file:
    txt_members = [
        m for m in zip_file.infolist()
        if m.filename.lower().endswith(".txt") and not m.is_dir()
    ]

    if not txt_members:
        raise SystemExit("ZIP failā nav atrasts neviens .txt fails.")

    print(f"\nAtrasti {len(txt_members)} .txt faili ZIP arhīvā.")
    print("Sākas failu apstrāde...\n")

    for i, member in enumerate(sorted(txt_members, key=lambda m: m.filename), start=1):
        try:
            raw_bytes = zip_file.read(member.filename)
            raw_text = decode_bytes(raw_bytes)

            year = extract_year(raw_text, fallback_filename=member.filename)
            doc_id = extract_id(raw_text, fallback_filename=member.filename)

            # Skaitām tikai 1934–1939 tabulai vajadzīgos dokumentus
            if year not in TARGET_YEARS:
                failed_rows.append({
                    "zip_filename": member.filename,
                    "id": doc_id,
                    "year": year,
                    "reason": "year_outside_target_or_missing"
                })
                continue

            # 1) LCLC vārdu/rakstzīmju skaitīšana
            text_body = extract_text_body_for_word_count(raw_bytes)
            word_metrics = count_lclc_word_metrics(text_body)

            # 2) TF–IDF/NMF priekšapstrāde un lematizēto vienību skaits
            clean_text = lemmatize_text_like_nmf(raw_text)

            if not clean_text.strip():
                failed_rows.append({
                    "zip_filename": member.filename,
                    "id": doc_id,
                    "year": year,
                    "reason": "empty_after_lemmatization"
                })
                continue

            lemma_unit_count = len(clean_text.split())

            rows.append({
                "id": doc_id,
                "zip_filename": member.filename,
                "meta_file_name": get_metadata_field(raw_text, "file_name"),
                "date": get_metadata_field(raw_text, "date"),
                "year": year,
                "words": word_metrics["words"],
                "chars_with_spaces": word_metrics["chars_with_spaces"],
                "chars_without_spaces": word_metrics["chars_without_spaces"],
                "lines": word_metrics["lines"],
                "lemmatized_units_after_preprocessing": lemma_unit_count,
                "clean_text": clean_text
            })

            if i % 10 == 0 or i == len(txt_members):
                print(f"Apstrādāti {i}/{len(txt_members)} faili; iekļauti tabulā: {len(rows)}")

        except Exception as e:
            failed_rows.append({
                "zip_filename": member.filename,
                "id": None,
                "year": None,
                "reason": f"exception: {e}"
            })

per_file_df = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed_rows)

print("\nIekļautie dokumenti:", len(per_file_df))
print("Neiekļautie / noraidītie dokumenti:", len(failed_df))

if per_file_df.empty:
    raise SystemExit("Nav neviena 1934–1939 dokumenta, ko iekļaut tabulā.")

# ============================================================
# 9. Kopsavilkuma tabula pa gadiem
# ============================================================

year_stats = (
    per_file_df
    .groupby("year")
    .agg(
        document_count=("zip_filename", "count"),
        words=("words", "sum"),
        chars_with_spaces=("chars_with_spaces", "sum"),
        chars_without_spaces=("chars_without_spaces", "sum"),
        avg_words_per_leaflet=("words", "mean"),
        median_words_per_leaflet=("words", "median"),
        lemmatized_units_after_preprocessing=("lemmatized_units_after_preprocessing", "sum")
    )
    .reindex(TARGET_YEARS, fill_value=0)
)

year_stats["avg_words_per_leaflet"] = year_stats["avg_words_per_leaflet"].round(2)
year_stats["median_words_per_leaflet"] = year_stats["median_words_per_leaflet"].round(0).astype(int)

total_docs = int(year_stats["document_count"].sum())
total_words = int(year_stats["words"].sum())

total_row = pd.DataFrame({
    "document_count": [total_docs],
    "words": [total_words],
    "chars_with_spaces": [int(year_stats["chars_with_spaces"].sum())],
    "chars_without_spaces": [int(year_stats["chars_without_spaces"].sum())],
    "avg_words_per_leaflet": [round(total_words / total_docs, 2) if total_docs else 0],
    "median_words_per_leaflet": [int(per_file_df["words"].median()) if total_docs else 0],
    "lemmatized_units_after_preprocessing": [
        int(year_stats["lemmatized_units_after_preprocessing"].sum())
    ]
}, index=["Kopā"])

year_stats_with_total = pd.concat([year_stats, total_row])

# ============================================================
# 10. Latviska tabula Word/pielikumam
# ============================================================

table_for_word = year_stats_with_total.rename(columns={
    "document_count": "Skrejlapu skaits",
    "words": "Vārdu skaits",
    "chars_with_spaces": "Rakstzīmes ar atstarpēm",
    "chars_without_spaces": "Rakstzīmes bez atstarpēm",
    "avg_words_per_leaflet": "Vidējais vārdu skaits vienā skrejlapā",
    "median_words_per_leaflet": "Mediānas vārdu skaits vienā skrejlapā",
    "lemmatized_units_after_preprocessing": "Lematizēto vienību skaits pēc priekšapstrādes"
})

table_for_word.index.name = "Gads"

print("\n=== 3. tabulas dati ===")
display(table_for_word)

# ============================================================
# 11. Kontroles pārbaude pret gaidītajām vērtībām
# Ja dati un Stanza versija sakrīt, vajadzētu iegūt šo tabulu.
# ============================================================

expected = {
    "Skrejlapu skaits": 244,
    "Vārdu skaits": 143538,
    "Rakstzīmes ar atstarpēm": 1076328,
    "Rakstzīmes bez atstarpēm": 931738,
    "Lematizēto vienību skaits pēc priekšapstrādes": 96119,
}

print("\n=== Kontrole pret gaidītajām kopējām vērtībām ===")

for col, expected_value in expected.items():
    actual_value = int(table_for_word.loc["Kopā", col])
    status = "OK" if actual_value == expected_value else "ATŠĶIRAS"
    print(f"{col}: {actual_value} | gaidīts: {expected_value} | {status}")

# ============================================================
# 12. Saglabāšana Excel failā
# ============================================================

output_xlsx = "reproduced_table_3_corpus_size_tfidf_nmf_1934_1939.xlsx"

with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
    table_for_word.to_excel(writer, sheet_name="3_tabula")
    per_file_df.drop(columns=["clean_text"]).to_excel(writer, sheet_name="per_file_counts", index=False)
    failed_df.to_excel(writer, sheet_name="excluded_diagnostics", index=False)

print("\nExcel fails saglabāts:", output_xlsx)
files.download(output_xlsx)

Izmantotais ZIP fails: /content/LCLC_1934-1940_only_unique_text_leaflets.zip

Lejupielādē un inicializē Stanza latviešu valodas modeli...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.12.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: lv (Latvian)...
| Processor | Package       |
-----------------------------
| tokenize  | lvtb          |
| pos       | lvtb_nocharlm |
| lemma     | lvtb_nocharlm |
| pretrain  | conll17       |

INFO:stanza:File exists: /root/.cache/stanza/1.12.0/resources/lv/tokenize/lvtb.pt
INFO:stanza:File exists: /root/.cache/stanza/1.12.0/resources/lv/pos/lvtb_nocharlm.pt
INFO:stanza:File exists: /root/.cache/stanza/1.12.0/resources/lv/lemma/lvtb_nocharlm.pt
INFO:stanza:File exists: /root/.cache/stanza/1.12.0/resources/lv/pretrain/conll17.pt
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.12.0/resources



Atrasti 251 .txt faili ZIP arhīvā.
Sākas failu apstrāde...

Apstrādāti 10/251 faili; iekļauti tabulā: 10
Apstrādāti 20/251 faili; iekļauti tabulā: 20
Apstrādāti 30/251 faili; iekļauti tabulā: 30
Apstrādāti 40/251 faili; iekļauti tabulā: 40
Apstrādāti 50/251 faili; iekļauti tabulā: 50
Apstrādāti 60/251 faili; iekļauti tabulā: 60
Apstrādāti 70/251 faili; iekļauti tabulā: 70
Apstrādāti 80/251 faili; iekļauti tabulā: 80
Apstrādāti 90/251 faili; iekļauti tabulā: 90
Apstrādāti 100/251 faili; iekļauti tabulā: 100
Apstrādāti 110/251 faili; iekļauti tabulā: 110
Apstrādāti 120/251 faili; iekļauti tabulā: 120
Apstrādāti 130/251 faili; iekļauti tabulā: 130
Apstrādāti 140/251 faili; iekļauti tabulā: 140
Apstrādāti 150/251 faili; iekļauti tabulā: 150
Apstrādāti 160/251 faili; iekļauti tabulā: 160
Apstrādāti 170/251 faili; iekļauti tabulā: 170
Apstrādāti 180/251 faili; iekļauti tabulā: 180
Apstrādāti 190/251 faili; iekļauti tabulā: 190
Apstrādāti 200/251 faili; iekļauti tabulā: 200
Apstrādāti 210/25

,Skrejlapu skaits,Vārdu skaits,Rakstzīmes ar atstarpēm,Rakstzīmes bez atstarpēm,Vidējais vārdu skaits vienā skrejlapā,Mediānas vārdu skaits vienā skrejlapā,Lematizēto vienību skaits pēc priekšapstrādes
Gads,,,,,,,
1934,94,52343,395074,342451,556.84,526,34932
1935,72,41178,310958,269429,571.92,465,27655
1936,29,20138,150737,130366,694.41,650,13687
1937,15,9038,65974,56841,602.53,482,5846
1938,13,8783,64917,56083,675.62,537,5869
1939,21,12058,88668,76568,574.19,570,8130
Kopā,244,143538,1076328,931738,588.27,529,96119



=== Kontrole pret gaidītajām kopējām vērtībām ===
Skrejlapu skaits: 244 | gaidīts: 244 | OK
Vārdu skaits: 143538 | gaidīts: 143538 | OK
Rakstzīmes ar atstarpēm: 1076328 | gaidīts: 1076328 | OK
Rakstzīmes bez atstarpēm: 931738 | gaidīts: 931738 | OK
Lematizēto vienību skaits pēc priekšapstrādes: 96119 | gaidīts: 96119 | OK

Excel fails saglabāts: reproduced_table_3_corpus_size_tfidf_nmf_1934_1939.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>